In [ ]:
from document_extractor import extract_itac_report

doc_1_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report1/LS2502 - Final Draft R2.docx"
doc_2_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report2/LS2508 - Final Draft.docx"
out = extract_itac_report(doc_2_path, output="html", save_files=True)

In [2]:
out.keys()

dict_keys(['general_information', 'annual_energy_usages_and_costs', 'carbon_footprint', 'recommendation_summary_table', 'ar_summary', 'assessment_recommendations'])

In [3]:
ar_summary = out['ar_summary']

ar_summary

'<p><i>AR No. 1 –\xa0</i><i>Utilize Higher Efficiency Lamps and/or Ballasts</i></p>\n<p>Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy efficiency measure will yield $771 in total cost savings and incur an implementation cost of $675. The project has a payback period of 0.88 years and will reduce CO₂ emissions by 3 tons annually, with annual energy savings of 6,806 kWh.</p>\n<p><i>AR No. 2 –\xa0</i><i>Install Sub-metering Equipment</i></p>\n<p>Installing sub-metering equipment will enable better monitoring and management of electrical energy consumption. This energy efficiency measure will yield $3,313 in total cost savings and incur an implementation cost of $3,000. The project has a payback period of 0.91 years and will reduce CO₂ emissions by 12 tons annually, with annual energy savings of 32,484 kWh.</p>\n<p><i>AR No. 3 – </i><i>Modify Inventory Control</i></p>\n<p>Modifying inventory control 

In [4]:
def parse_ar_summaries(ar_summary_html):
    """
    Parse AR summary HTML and return a list of dictionaries with AR number and summary.
    
    Args:
        ar_summary_html (str): HTML string containing AR summaries
        
    Returns:
        List[Dict]: List of dictionaries with 'ar_no' (int) and 'ar_summary' (str) keys
    """
    import re
    from bs4 import BeautifulSoup
    
    # Parse the HTML
    soup = BeautifulSoup(ar_summary_html, 'html.parser')
    
    # Find all paragraphs
    paragraphs = soup.find_all('p')
    
    ar_summaries = []
    current_ar = None
    current_summary_parts = []
    
    for p in paragraphs:
        text = p.get_text()
        
        # Check if this paragraph starts with "AR No. X"
        ar_match = re.match(r'AR No\.\s*(\d+)', text, re.IGNORECASE)
        
        if ar_match:
            # If we have a previous AR, save it
            if current_ar is not None and current_summary_parts:
                ar_summaries.append({
                    'ar_no': current_ar,
                    'ar_summary': ' '.join(current_summary_parts).strip()
                })
            
            # Start new AR
            current_ar = int(ar_match.group(1))
            current_summary_parts = [text]
        else:
            # This is a continuation of the current AR summary
            if current_ar is not None:
                current_summary_parts.append(text)
    
    # Don't forget the last AR
    if current_ar is not None and current_summary_parts:
        ar_summaries.append({
            'ar_no': current_ar,
            'ar_summary': ' '.join(current_summary_parts).strip()
        })
    
    return ar_summaries

# Test the function
parsed_summaries = parse_ar_summaries(ar_summary)
print(f"Found {len(parsed_summaries)} AR summaries:")
for summary in parsed_summaries:
    print(f"AR {summary['ar_no']}: {summary['ar_summary'][:100]}...")


Found 8 AR summaries:
AR 1: AR No. 1 – Utilize Higher Efficiency Lamps and/or Ballasts Utilizing higher-efficiency lamps and/or ...
AR 2: AR No. 2 – Install Sub-metering Equipment Installing sub-metering equipment will enable better monit...
AR 3: AR No. 3 – Modify Inventory Control Modifying inventory control systems will streamline administrati...
AR 4: AR No. 4 – Replace Existing HVAC with Higher Efficiency Model Replacing the existing HVAC system wit...
AR 5: AR No. 5 – Use Solar Heat to Generate Electricity Installing a solar heat system for electricity gen...
AR 6: AR No. 6 – Consider Replacement of Old Motors with Energy-Efficient Ones Replacing old motors with e...
AR 7: AR No.7  – Purchase Optimum Sized Air Compressor with More Suitable Substitutes Purchasing an optima...
AR 8: AR No. 8 – Replace Fossil Fuel Equipment with Electrical Equipment Replacing fossil fuel equipment w...


In [6]:
parsed_summaries[1]

{'ar_no': 2,
 'ar_summary': 'AR No. 2 –\xa0Install Sub-metering Equipment Installing sub-metering equipment will enable better monitoring and management of electrical energy consumption. This energy efficiency measure will yield $3,313 in total cost savings and incur an implementation cost of $3,000. The project has a payback period of 0.91 years and will reduce CO₂ emissions by 12 tons annually, with annual energy savings of 32,484 kWh.'}